## Combine raw gaze data and pupil diameter data 

In [1]:
from pynwb import NWBHDF5IO
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np 
import os
from scipy.stats import pearsonr
from scipy.stats import gaussian_kde
import cv2
from scipy.signal import butter, filtfilt
from scipy.signal import savgol_filter

from scipy.ndimage import gaussian_filter
from pathlib import Path

# Set working directory to test this out locally 
os.chdir('/Users/nicole.burke/OneDrive - Child Mind Institute/02_Projects/06_rockland_sample/01_rockland_descriptor_paper/complied_server_data')
print(os.getcwd())

### Custom functions for script 
def descr_stats(df, column_name):
    stats = df[column_name].agg(
        mean='mean',
        min='min',
        max='max',
        std='std'
    )
    print(f"Summary stats for column: {column_name}")
    print(stats)

    return stats

/Users/nicole.burke/Library/CloudStorage/OneDrive-ChildMindInstitute/02_Projects/06_rockland_sample/01_rockland_descriptor_paper/complied_server_data


### Import data - *Present* DS1

In [10]:
#### Read in data 
present_df_raw_gaze = pd.read_csv('present_ds1_df.csv')
present_df_raw_gaze = present_df_raw_gaze.iloc[:, 1:]
print("PRESENT DS1: RAW GAZE")
print(present_df_raw_gaze.head())
print(present_df_raw_gaze.shape)

print("*"*40)
print("PRESENT DS1: PUPIL")
present_df_pupil = pd.read_csv('present_ds1_pupil.csv')
present_df_pupil = present_df_pupil.iloc[:, 1:]
print(present_df_pupil.head())
print(present_df_pupil.shape)
print("col1", present_df_pupil.columns[0])
print("col2",present_df_pupil.columns[1])
print("col3", present_df_pupil.columns[2])
print("col4",present_df_pupil.columns[3])

PRESENT DS1: RAW GAZE
   x_corr_pixels  y_corr_pixels     times      subjectID
0          564.2          631.6 -1.122725  sub-A00010201
1          564.3          630.9 -1.117170  sub-A00010201
2          564.5          629.8 -1.111614  sub-A00010201
3          564.7          628.0 -1.106058  sub-A00010201
4          564.3          628.4 -1.100503  sub-A00010201
(7500629, 4)
****************************************
PRESENT DS1: PUPIL
   Pupil diameter(mm) extracted from both eyes. [left eye   right eye]  \
0                                               5.44              4.78   
1                                               5.44              4.78   
2                                               5.44              4.78   
3                                               5.45              4.78   
4                                               5.45              4.78   

      times      subjectID  
0 -1.122725  sub-A00010201  
1 -1.117170  sub-A00010201  
2 -1.111614  sub-A00010201  
3 

In [ ]:
### Fix column heads on present_df_pupil 
present_df_pupil = present_df_pupil.rename(
    columns={
        "Pupil diameter(mm) extracted from both eyes. [left eye": "left_eye_pupil_mm",
        " right eye]": "right_eye_pupil_mm",
        "times": "times",
        "subjectID": "subjectID",
    }
)
print(present_df_pupil.head())

### Merge data 
present_df_all_raw = pd.merge(
    present_df_raw_gaze,
    present_df_pupil,
    on=["subjectID", "times"],
    how="inner",
)

print(present_df_all_raw.head())
print(present_df_all_raw.shape)

   x_corr_pixels  y_corr_pixels     times      subjectID  left_eye_pupil_mm  \
0          564.2          631.6 -1.122725  sub-A00010201               5.44   
1          564.3          630.9 -1.117170  sub-A00010201               5.44   
2          564.5          629.8 -1.111614  sub-A00010201               5.44   
3          564.7          628.0 -1.106058  sub-A00010201               5.45   
4          564.3          628.4 -1.100503  sub-A00010201               5.45   

   right_eye_pupil_mm  
0                4.78  
1                4.78  
2                4.78  
3                4.78  
4                4.78  
(7500629, 6)


### Import data - *Present* DS2

In [20]:
#### Read in data 
present_df2_raw_gaze = pd.read_csv('present_ds2_right_eye_df.csv')
present_df2_raw_gaze = present_df2_raw_gaze.iloc[:, 1:]
print("PRESENT DS2: RAW GAZE")
print(present_df2_raw_gaze.head())
print(present_df2_raw_gaze.shape)

print("*"*40)
print("PRESENT DS2: PUPIL")
present_df2_pupil = pd.read_csv('present_ds2_pupil.csv')
present_df2_pupil = present_df2_pupil.iloc[:, 1:]
print(present_df2_pupil.head())
print(present_df2_pupil.shape)
print("col1", present_df2_pupil.columns[0])
print("col2",present_df2_pupil.columns[1])
print("col3", present_df2_pupil.columns[2])
print("col4",present_df2_pupil.columns[3])

PRESENT DS2: RAW GAZE
    rightEyeX   rightEyeY     times      subjectID
0  629.099976  518.599976 -1.367178  sub-A00008326
1  629.700012  519.400024 -1.365070  sub-A00008326
2  629.700012  519.400024 -1.363065  sub-A00008326
3  627.400024  524.000000 -1.361112  sub-A00008326
4  623.799988  529.200012 -1.359141  sub-A00008326
(12527062, 4)
****************************************
PRESENT DS2: PUPIL
   leftPupilArea  rightPupilArea     times      subjectID
0            0.0           320.0 -1.367178  sub-A00008326
1            0.0           317.0 -1.365070  sub-A00008326
2            0.0           312.0 -1.363065  sub-A00008326
3            0.0           309.0 -1.361112  sub-A00008326
4            0.0           307.0 -1.359141  sub-A00008326
(12527062, 4)
col1 leftPupilArea
col2 rightPupilArea
col3 times
col4 subjectID


In [21]:
### Merge data 
present_df2_all_raw = pd.merge(
    present_df2_raw_gaze,
    present_df2_pupil,
    on=["subjectID", "times"],
    how="inner",
)

print(present_df2_all_raw.head())
print(present_df2_all_raw.shape)

    rightEyeX   rightEyeY     times      subjectID  leftPupilArea  \
0  629.099976  518.599976 -1.367178  sub-A00008326            0.0   
1  629.700012  519.400024 -1.365070  sub-A00008326            0.0   
2  629.700012  519.400024 -1.363065  sub-A00008326            0.0   
3  627.400024  524.000000 -1.361112  sub-A00008326            0.0   
4  623.799988  529.200012 -1.359141  sub-A00008326            0.0   

   rightPupilArea  
0           320.0  
1           317.0  
2           312.0  
3           309.0  
4           307.0  
(12527062, 6)


### Import data - *Sherlock* DS1

In [23]:
#### Read in data 
sherlock_df_raw_gaze = pd.read_csv('sherlock_ds1_df.csv')
sherlock_df_raw_gaze = sherlock_df_raw_gaze.iloc[:, 1:]
print("SHERLOCK DS1: RAW GAZE")
print(sherlock_df_raw_gaze.head())
print(sherlock_df_raw_gaze.shape)

print("*"*40)
print("SHERLOCK DS1: PUPIL")
sherlock_df_pupil = pd.read_csv('sherlock_ds1_pupil.csv')
sherlock_df_pupil = sherlock_df_pupil.iloc[:, 1:]
print(sherlock_df_pupil.head())
print(sherlock_df_pupil.shape)
print("col1", sherlock_df_pupil.columns[0])
print("col2",sherlock_df_pupil.columns[1])
print("col3", sherlock_df_pupil.columns[2])
print("col4",sherlock_df_pupil.columns[3])

SHERLOCK DS1: RAW GAZE
   x_corr_pixels  y_corr_pixels     times      subjectID
0          669.9          513.0 -0.405907  sub-A00010201
1          669.9          512.3 -0.400351  sub-A00010201
2          670.0          511.9 -0.394795  sub-A00010201
3          670.1          511.2 -0.389240  sub-A00010201
4          670.0          510.8 -0.383684  sub-A00010201
(20762676, 4)
****************************************
SHERLOCK DS1: PUPIL
   Pupil diameter(mm) extracted from both eyes. [left eye   right eye]  \
0                                               5.41              4.89   
1                                               5.41              4.89   
2                                               5.40              4.88   
3                                               5.40              4.88   
4                                               5.40              4.88   

      times      subjectID  
0 -0.405907  sub-A00010201  
1 -0.400351  sub-A00010201  
2 -0.394795  sub-A00010201  

In [25]:
### Fix column heads on sherlock_df_pupil 
sherlock_df_pupil = sherlock_df_pupil.rename(
    columns={
        "Pupil diameter(mm) extracted from both eyes. [left eye": "left_eye_pupil_mm",
        " right eye]": "right_eye_pupil_mm",
        "times": "times",
        "subjectID": "subjectID",
    }
)
print(sherlock_df_pupil.head())

### Merge data 
sherlock_df_all_raw = pd.merge(
    sherlock_df_raw_gaze,
    sherlock_df_pupil,
    on=["subjectID", "times"],
    how="inner",
)

print(sherlock_df_all_raw.head())
print(sherlock_df_all_raw.shape)

   left_eye_pupil_mm  right_eye_pupil_mm     times      subjectID
0               5.41                4.89 -0.405907  sub-A00010201
1               5.41                4.89 -0.400351  sub-A00010201
2               5.40                4.88 -0.394795  sub-A00010201
3               5.40                4.88 -0.389240  sub-A00010201
4               5.40                4.88 -0.383684  sub-A00010201
   x_corr_pixels  y_corr_pixels     times      subjectID  left_eye_pupil_mm  \
0          669.9          513.0 -0.405907  sub-A00010201               5.41   
1          669.9          512.3 -0.400351  sub-A00010201               5.41   
2          670.0          511.9 -0.394795  sub-A00010201               5.40   
3          670.1          511.2 -0.389240  sub-A00010201               5.40   
4          670.0          510.8 -0.383684  sub-A00010201               5.40   

   right_eye_pupil_mm  
0                4.89  
1                4.89  
2                4.88  
3                4.88  
4        

### Import data - *Sherlock* DS2

In [27]:
#### Read in data 
sherlock_df2_raw_gaze = pd.read_csv('sherlock_ds2_right_eye_df.csv')
sherlock_df2_raw_gaze = sherlock_df2_raw_gaze.iloc[:, 1:]
print("SHERLOCK DS2: RAW GAZE")
print(sherlock_df2_raw_gaze.head())
print(sherlock_df2_raw_gaze.shape)

print("*"*40)
print("SHERLOCK DS2: PUPIL")
sherlock_df2_pupil = pd.read_csv('sherlock_ds2_pupil.csv')
sherlock_df2_pupil = sherlock_df2_pupil.iloc[:, 1:]
print(sherlock_df2_pupil.head())
print(sherlock_df2_pupil.shape)
print("col1", sherlock_df2_pupil.columns[0])
print("col2",sherlock_df2_pupil.columns[1])
print("col3", sherlock_df2_pupil.columns[2])
print("col4",sherlock_df2_pupil.columns[3])

SHERLOCK DS2: RAW GAZE
    rightEyeX   rightEyeY     times      subjectID
0  884.000000  529.299988 -1.433650  sub-A00008326
1  895.900024  519.099976 -1.433295  sub-A00008326
2  897.099976  507.399994 -1.431131  sub-A00008326
3  896.400024  507.600006 -1.429064  sub-A00008326
4  896.500000  506.799988 -1.427053  sub-A00008326
(35309166, 4)
****************************************
SHERLOCK DS2: PUPIL
   leftPupilArea  rightPupilArea     times      subjectID
0            0.0           267.0 -1.433650  sub-A00008326
1            0.0           287.0 -1.433295  sub-A00008326
2            0.0           300.0 -1.431131  sub-A00008326
3            0.0           301.0 -1.429064  sub-A00008326
4            0.0           302.0 -1.427053  sub-A00008326
(35309166, 4)
col1 leftPupilArea
col2 rightPupilArea
col3 times
col4 subjectID


In [28]:
### Merge data 
sherlock_df2_all_raw = pd.merge(
    sherlock_df2_raw_gaze,
    sherlock_df2_pupil,
    on=["subjectID", "times"],
    how="inner",
)

print(sherlock_df2_all_raw.head())
print(sherlock_df2_all_raw.shape)

    rightEyeX   rightEyeY     times      subjectID  leftPupilArea  \
0  884.000000  529.299988 -1.433650  sub-A00008326            0.0   
1  895.900024  519.099976 -1.433295  sub-A00008326            0.0   
2  897.099976  507.399994 -1.431131  sub-A00008326            0.0   
3  896.400024  507.600006 -1.429064  sub-A00008326            0.0   
4  896.500000  506.799988 -1.427053  sub-A00008326            0.0   

   rightPupilArea  
0           267.0  
1           287.0  
2           300.0  
3           301.0  
4           302.0  
(35309234, 6)


In [29]:
sherlock_df2_all_raw.to_csv('sherlock_df2_all_raw.csv')